In [8]:
# Load the dataset
from pyspark.sql import SparkSession
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"

# Initialize Spark Session
spark = SparkSession.builder.appName("LightcastData").getOrCreate()

# Load Data
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("data/lightcast_job_postings.csv")

# Show Schema and Sample Data
df.printSchema()
df.show(5)
df.createOrReplaceTempView("lightcast_job_postings")



root
 |-- ID: string (nullable = true)
 |-- LAST_UPDATED_DATE: string (nullable = true)
 |-- LAST_UPDATED_TIMESTAMP: timestamp (nullable = true)
 |-- DUPLICATES: integer (nullable = true)
 |-- POSTED: string (nullable = true)
 |-- EXPIRED: string (nullable = true)
 |-- DURATION: integer (nullable = true)
 |-- SOURCE_TYPES: string (nullable = true)
 |-- SOURCES: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- ACTIVE_URLS: string (nullable = true)
 |-- ACTIVE_SOURCES_INFO: string (nullable = true)
 |-- TITLE_RAW: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- MODELED_EXPIRED: string (nullable = true)
 |-- MODELED_DURATION: integer (nullable = true)
 |-- COMPANY: integer (nullable = true)
 |-- COMPANY_NAME: string (nullable = true)
 |-- COMPANY_RAW: string (nullable = true)
 |-- COMPANY_IS_STAFFING: boolean (nullable = true)
 |-- EDUCATION_LEVELS: string (nullable = true)
 |-- EDUCATION_LEVELS_NAME: string (nullable = true)
 |-- MIN_EDULEVELS: integer (

In [20]:
df=df.dropna(subset=['salary'])
df=df[df['salary']!=0]
df.createOrReplaceTempView("lightcast_job_postings")

In [24]:
employ_df=spark.sql("""
SELECT naics_2022_6_name, 
    COUNT(*) AS employ_count,
    SUM(salary) AS tot_salary,
    AVG(salary) AS avg_salary,
    MIN(salary) AS min_salary,
    MAX(salary) AS max_salary,
    STDDEV(salary) AS std_salary
FROM lightcast_job_postings
GROUP BY naics_2022_6_name
ORDER BY avg_salary DESC
""")
employ_df.show(5)
employ_df.createOrReplaceTempView("employ_df")

+--------------------+------------+----------+-----------------+----------+----------+------------------+
|   naics_2022_6_name|employ_count|tot_salary|       avg_salary|min_salary|max_salary|        std_salary|
+--------------------+------------+----------+-----------------+----------+----------+------------------+
|Polish and Other ...|           1|    217050|         217050.0|    217050|    217050|              NULL|
|     Travel Agencies|          17|   3515642|206802.4705882353|     71000|    353500|116273.49153510746|
|Battery Manufactu...|           1|    205982|         205982.0|    205982|    205982|              NULL|
|Petrochemical Man...|           1|    202500|         202500.0|    202500|    202500|              NULL|
|Rubber and Plasti...|           1|    195000|         195000.0|    195000|    195000|              NULL|
+--------------------+------------+----------+-----------------+----------+----------+------------------+
only showing top 5 rows


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
pandas_employ=employ_df.toPandas()
pandas_employ.plot(x='naics_2022_6_name', y='salary_from', kind='bar', color='skyblue', title='Salary Distribution by Employment Type') 
plt.xlabel('Employment Type Name')
plt.ylabel('Salary From') 
plt.show()

In [34]:
posted_df=spark.sql("""
SELECT posted,
    COUNT(*) AS post_count
FROM lightcast_job_postings
GROUP BY posted
ORDER BY posted
""")
posted_df.show(5)

+---------+----------+
|   posted|post_count|
+---------+----------+
| 5/1/2024|       139|
|5/10/2024|       246|
|5/11/2024|       242|
|5/12/2024|       449|
|5/13/2024|       123|
+---------+----------+
only showing top 5 rows
